In [ ]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    "unsloth/llama-2-7b-bnb-4bit",
    "unsloth/llama-2-13b-bnb-4bit",
    "unsloth/codellama-34b-bnb-4bit",
    "unsloth/tinyllama-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit", # New Google 6 trillion tokens model 2.5x faster!
    "unsloth/gemma-2b-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit", # Choose ANY! eg teknium/OpenHermes-2.5-Mistral-7B
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = "", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.1.5: Fast Mistral patching. Transformers: 4.47.1.
   \\   /|    GPU: NVIDIA A100-SXM4-40GB. Max memory: 39.564 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 8.0. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.1.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
import pandas as pd

df = pd.read_parquet("/content/med_dataset.parquet")
df.head()

,conversations
0,"[{'from': 'human', 'value': 'What are the side..."
1,"[{'from': 'human', 'value': 'What are the side..."
2,"[{'from': 'human', 'value': 'What are the side..."
3,"[{'from': 'human', 'value': 'What are the side..."
4,"[{'from': 'human', 'value': 'What are the side..."


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # Supports zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, unsloth
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # ShareGPT style
    map_eos_token = True, # Maps <|im_end|> to </s> instead
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }
pass

from datasets import load_dataset
dataset = load_dataset("efedemircan/med_data", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

Unsloth: Will map <|im_end|> to EOS = </s>.
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

med_dataset.parquet:   0%|          | 0.00/49.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/248218 [00:00<?, ? examples/s]

Map:   0%|          | 0/248218 [00:00<?, ? examples/s]

In [ ]:
dataset[5]["conversations"]

[{'from': 'human', 'value': 'What are the side effects of allegra-m tablet?'},
 {'from': 'gpt',
  'value': 'The side effects of allegra-m tablet are: Nausea, Diarrhea, Vomiting, Skin rash, Flu-like symptoms, Headache, Drowsiness, Dizziness'},
 {'from': 'human', 'value': 'What are the uses of allegra-m tablet?'},
 {'from': 'gpt',
  'value': 'allegra-m tablet is used for: Treatment of Sneezing and runny nose due to allergies'},
 {'from': 'human',
  'value': 'What are the alternative medications for allegra-m tablet?'},
 {'from': 'gpt',
  'value': 'The alternative medications for allegra-m tablet are: Emlukast-FX Tablet, LCFEX-Mont Tablet, Fixar 10mg/120mg Tablet, Histakind-M Tablet, Histafree-M Tablet'},
 {'from': 'human',
  'value': 'Can you provide general information about allegra-m tablet?'},
 {'from': 'gpt',
  'value': 'allegra-m tablet belongs to the RESPIRATORY class. Mechanism of action: . Chemical class: . This medication is not habit-forming.'}]

In [ ]:
print(dataset[5]["text"])

<|im_start|>user
What are the side effects of allegra-m tablet?<|im_end|>
<|im_start|>assistant
The side effects of allegra-m tablet are: Nausea, Diarrhea, Vomiting, Skin rash, Flu-like symptoms, Headache, Drowsiness, Dizziness<|im_end|>
<|im_start|>user
What are the uses of allegra-m tablet?<|im_end|>
<|im_start|>assistant
allegra-m tablet is used for: Treatment of Sneezing and runny nose due to allergies<|im_end|>
<|im_start|>user
What are the alternative medications for allegra-m tablet?<|im_end|>
<|im_start|>assistant
The alternative medications for allegra-m tablet are: Emlukast-FX Tablet, LCFEX-Mont Tablet, Fixar 10mg/120mg Tablet, Histakind-M Tablet, Histafree-M Tablet<|im_end|>
<|im_start|>user
Can you provide general information about allegra-m tablet?<|im_end|>
<|im_start|>assistant
allegra-m tablet belongs to the RESPIRATORY class. Mechanism of action: . Chemical class: . This medication is not habit-forming.<|im_end|>



In [ ]:
unsloth_template = \
    "{{ bos_token }}"\
    "{{ 'You are a helpful assistant to the user\n' }}"\
    "{% for message in messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ '>>> User: ' + message['content'] + '\n' }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ '>>> Assistant: ' + message['content'] + eos_token + '\n' }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}"\
        "{{ '>>> Assistant: ' }}"\
    "{% endif %}"
unsloth_eos_token = "eos_token"

if False:
    tokenizer = get_chat_template(
        tokenizer,
        chat_template = (unsloth_template, unsloth_eos_token,), # You must provide a template and EOS token
        mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # ShareGPT style
        map_eos_token = True, # Maps <|im_end|> to </s> instead
    )

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Map (num_proc=2):   0%|          | 0/248218 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 248,218 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 60
 "-____-"     Number of trainable parameters = 41,943,040


Step,Training Loss
1,1.766000
2,1.676600
3,1.644500
4,1.436200
5,1.110000
6,0.929400
7,0.798500
8,0.701500
9,0.630000
10,0.676700


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # Supports zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, unsloth
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # ShareGPT style
    map_eos_token = True, # Maps <|im_end|> to </s> instead
)

FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"from": "human", "value": "What are the side effects of allegra 120mg,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


['<|im_start|>user\nWhat are the side effects of allegra 120mg,<|im_end|>\n<|im_start|>assistant\nThe side effects of allegra 120mg are: Nausea, Vomiting, Diarrhea, Stomach pain, Dryness in mouth, Headache, Dizziness, Sleepiness, Fatigue, Weakness<|im_end|>']

In [ ]:
model.push_to_hub("efedemircan/drug-llm-v0.1", token = "") # Online saving
tokenizer.push_to_hub("efedemircan/drug-llm-tokenizer", token = "") # Online saving

README.md:   0%|          | 0.00/610 [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Saved model to https://huggingface.co/efedemircan/drug-llm-v0.1


In [ ]:
model.save_pretrained("drug-llm-v0.1")  # Local saving
tokenizer.save_pretrained("drug-llm-tokenizer") # Local saving

('drug-llm-tokenizer/tokenizer_config.json',
 'drug-llm-tokenizer/special_tokens_map.json',
 'drug-llm-tokenizer/tokenizer.json')

In [ ]:
!pip install --upgrade streamlit

In [ ]:
%%writefile app.py
import streamlit as st
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch

# Sayfa başlığı ve açıklama
st.title("İlaç Bilgi Asistanı")
st.write("Bu uygulama, ilaçlar hakkında bilgi almanıza yardımcı olur.")

# Önceden eğitilmiş modeli ve tokenizer'ı yükleyin
@st.cache_resource
def load_model_and_tokenizer():
    model, tokenizer = FastLanguageModel.from_pretrained(
        "efedemircan/drug-llm-v0.1",  # Pretrain edilmiş modelinizin adı
        load_in_4bit=True,  # 4-bit quantizasyonu kullanın (isteğe bağlı)
    )
    tokenizer = get_chat_template(
        tokenizer,
        chat_template="chatml",
        mapping={"role": "from", "content": "value", "user": "human", "assistant": "gpt"},
        map_eos_token=True,
    )
    return model, tokenizer

model, tokenizer = load_model_and_tokenizer()

# Kullanıcı girişi
user_input = st.text_area("Sormak istediğiniz soruyu yazın:", height=100)

if st.button("Cevapla"):
    if user_input:
        with st.spinner("Cevap hazırlanıyor..."):
            # Mesajı formatlama
            messages = [{"from": "human", "value": user_input}]

            # Chat template'i uygula
            inputs = tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
            ).to(model.device)  # Modelin cihazına taşı

            # Çıktı oluştur
            outputs = model.generate(input_ids=inputs, max_new_tokens=256, use_cache=True)

            # Yanıtı decode et
            response = tokenizer.batch_decode(outputs)[0]

            # Yanıtı göster
            st.write("### Yanıt:")
            st.write(response)

    else:
        st.warning("Lütfen bir soru girin.")

Writing app.py


In [ ]:
!pip install streamlit

  Using cached streamlit-1.41.1-py2.py3-none-any.whl.metadata (8.5 kB)
Using cached streamlit-1.41.1-py2.py3-none-any.whl (9.1 MB)


In [ ]:
!streamlit run app.py --server.port 8502

Usage: streamlit run [OPTIONS] TARGET [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value for '--server.port': '0.0.0.0' is not a valid integer.


In [ ]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.2 MB/s eta 0:00:00
